# Jour 03: Stockage des une base de donnees et visualiation des Vecteurs (ensemble de centaine de nombre stocker le sens ou la semantique d'un text)

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

In [1]:
# imports

import os
import glob
import tiktoken
import numpy as np
# import sentence_transformers 
import plotly.graph_objects as go

from sklearn.manifold import TSNE
from dotenv import load_dotenv
from typing import List

# Importation des bibliothèques de LangChain
from langchain_chroma import Chroma
from langchain_postgres import PGVector
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.embeddings import FastEmbedEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader, DirectoryLoader


In [2]:
# Charger les variables d'environnement
load_dotenv()

True

In [ ]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

# Load environment variables in a file called .env

load_dotenv()
openai_api_key = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

## Partie A: Diviser le document en morceaux

In [3]:
# Combien de parametre il y a dasn chaque documents 

knowledge_base_path = "knowledge-base/**/*.md"
files = glob.glob(knowledge_base_path, recursive=True)
print(f"Found {len(files)} files in the knowledge base.")

entire_knowledge_base = ""

for file_path in files:
    with open(file_path, 'r', encoding='utf-8') as f:
        entire_knowledge_base = f.read()
        entire_knowledge_base += "\n\n"
        
print(f"Total characters in the knowledge base: {len(entire_knowledge_base):,}")

Found 31 files in the knowledge base.
Total characters in the knowledge base: 223


In [ ]:
# Combien de token compte tous les documents

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)

print(f"Le total des tokens pour le modle {MODEL} est : {len(tokens):,}")

In [4]:
# Charger les documents de la base de connaissances et ajouter un champ de métadonnées pour le type de document
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

In [5]:
len(documents)

31

In [6]:
documents[24]

Document(metadata={'source': 'knowledge-base/products/Rellm.md', 'doc_type': 'products'}, page_content="# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intellige

In [7]:
# Diviser les document en morceaux
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divised into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divised into 123 chunks
First chunk:

page_content='# HR Record

# Jordan Blake

## Summary
- **Date of Birth:** March 15, 1993  
- **Job Title:** Sales Development Representative (SDR)  
- **Location:** Austin, Texas  

## Insurellm Career Progression
- **2021-06:** Joined Insurellm as an Entry-Level SDR  
- **2022-02:** Promoted to Junior SDR after exceeding quarterly targets by 25%  
- **2022-12:** Recognized as SDR of the Month for three consecutive months  
- **2023-05:** Participated in the Insurellm Leadership Training Program' metadata={'source': 'knowledge-base/employees/Jordan Blake.md', 'doc_type': 'employees'}


In [8]:
len(chunks)

123

In [9]:
chunks[6]

Document(metadata={'source': 'knowledge-base/employees/Alex Chen.md', 'doc_type': 'employees'}, page_content='Alex Chen continues to be a vital asset at Insurellm, contributing significantly to innovative backend solutions that help shape the future of insurance technology.')

## Partie B: Sauvegarder les vecteurs sous forme dans un base de donnee

### 1. Chroma BD et HuggingFace

In [ ]:
# Créer une base de données vectorielle avec Chroma et les embeddings de Hugging Face
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
    
vector_store = Chroma.from_documents(chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vector store created with {vector_store._collection.count()} documents")

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

In [ ]:
collection = vector_store._collection
count = collection.count()

sample_embeddings = collection.get(limit=1, include=["embeddings", "metadatas", "documents"])
dimensions = len(sample_embeddings["embeddings"][0])
print(f"There are {count:,} vectors with {dimensions} dimensions in the vector store.")

### 2. POSGRESQL avec PGVector et FastEmbeddings

In [10]:
# Tes accès (Idéalement à mettre dans ton fichier .env)
CONNECTION_STRING = os.getenv("DATABASE_URL")
COLLECTION_NAME = "insurance_knowledge_base"

# 1. Initialiser le modèle d'embedding (une fois l'étape 2 réglée)
embeddings = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

# 3. Création du Vector Store dans PostgreSQL
vector_store = PGVector.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    connection=CONNECTION_STRING,
    use_jsonb=True,
    pre_delete_collection=True 
)

print(f"Vector store créé dans PostgreSQL avec {len(chunks)} documents.")

Vector store créé dans PostgreSQL avec 123 documents.


In [11]:
# 1. Compter le nombre de vecteurs (via l'ID des documents stockés)
# Si tu viens juste de créer vector_store, tu peux utiliser len(chunks)
# Sinon, pour interroger la base réelle :
count = len(vector_store.get_by_ids(vector_store.ids)) if hasattr(vector_store, 'ids') else len(chunks)

# 2. Obtenir la dimension des vecteurs
# On demande au modèle d'embedding quelle est sa taille de sortie
sample_embedding = embeddings.embed_query("Test")
dimensions = len(sample_embedding)

print(f"Il y a {len(chunks):,} vecteurs avec {dimensions} dimensions dans PostgreSQL.")

Il y a 123 vecteurs avec 384 dimensions dans PostgreSQL.


## Partie C: Visualisation!

In [12]:
# 1. On extrait tout depuis PostgreSQL (via PGVector)
with vector_store.session_maker() as session:
    all_data = session.query(vector_store.EmbeddingStore).all()

# 2. On prépare nos listes pour Plotly et TSNE
vectors_list = [e.embedding for e in all_data]
documents_list = [e.document for e in all_data]
doc_types_list = [e.cmetadata.get('doc_type', 'unknown') for e in all_data]

vectors_np = np.array(vectors_list) # TSNE a besoin d'un tableau NumPy

# 3. Mapping des couleurs (Harmonisation)
color_map = {
    'products': 'blue', 
    'employees': 'red', 
    'contracts': 'green', 
    'company': 'yellow'
}
colors = [color_map.get(t, 'gray') for t in doc_types_list]

print(f"✅ {len(vectors_np)} vecteurs prêts pour la visualisation.")

✅ 123 vecteurs prêts pour la visualisation.


### En 2D

In [15]:
# T-SNE configuré pour 2 dimensions
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors_np)-1))
vectors_2d = tsne_2d.fit_transform(vectors_np)

# Création du graphique 2D
fig_2d = go.Figure(data=[go.Scatter(
    x=vectors_2d[:, 0],
    y=vectors_2d[:, 1],
    mode='markers',
    marker=dict(size=7, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Texte: {d[:100]}..." for t, d in zip(doc_types_list, documents_list)],
    hoverinfo='text'
)])

fig_2d.update_layout(
    title="2D Vector Store Visualization (PostgreSQL)", 
    xaxis_title='Axe X', 
    yaxis_title='Axe Y',
    width=800,
    height=600,
    margin=dict(l=0, r=0, b=0, t=30)
)

fig_2d.show()

### En 3D

In [16]:
# T-SNE configuré pour 3 dimensions
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(30, len(vectors_np)-1))
vectors_3d = tsne_3d.fit_transform(vectors_np)

# Création du graphique 3D (go.Scatter3d au lieu de go.Scatter)
fig_3d = go.Figure(data=[go.Scatter3d(
    x=vectors_3d[:, 0],
    y=vectors_3d[:, 1],
    z=vectors_3d[:, 2], # Ajout vital de l'axe Z
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Texte: {d[:100]}..." for t, d in zip(doc_types_list, documents_list)],
    hoverinfo='text'
)])

fig_3d.update_layout(
    title="3D Vector Store Visualization (PostgreSQL)", 
    scene=dict(xaxis_title='Axe X', yaxis_title='Axe Y', zaxis_title='Axe Z'),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=30)
)

fig_3d.show()